In [1]:
import subprocess
subprocess.run(["pip", "install", "-q", "albumentations", "--upgrade"])

CompletedProcess(args=['pip', 'install', '-q', 'albumentations', '--upgrade'], returncode=0)

In [2]:
import os, gc, re, math, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
 
import albumentations as A
from albumentations.pytorch import ToTensorV2
 
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
 
from PIL import Image
import cv2
 
warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True
 
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything()
 
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cuda


In [3]:
BASE_DIR = Path('/kaggle/input/multimodal-cancer-classification-challenge-2026')
 
BF_TRAIN = BASE_DIR / 'BF' / 'train'
BF_TEST  = BASE_DIR / 'BF' / 'test'
FL_TRAIN = BASE_DIR / 'FL' / 'train'
FL_TEST  = BASE_DIR / 'FL' / 'test'
 
TRAIN_CSV  = BASE_DIR / 'train.csv'
SAMPLE_SUB = BASE_DIR / 'sampleSubmission.csv'
OUT_DIR    = Path('/kaggle/working')
 
CFG = dict(
    img_size     = 224,
    batch_size   = 64,
    n_epochs     = 20,
    lr           = 3e-4,
    weight_decay = 1e-2,
    n_folds      = 5,
    label_smooth = 0.05,
    mixup_alpha  = 0.4,
    tta_n        = 5,
    amp          = True,
)

In [4]:
import kagglehub
 
_download_path = kagglehub.competition_download(
    'multimodal-cancer-classification-challenge-2026')
print("Competition files downloaded to:", _download_path)
 
BASE_DIR = Path(_download_path)
 
# kagglehub sometimes nests files one level deeper; auto-detect
if not (BASE_DIR / 'train.csv').exists():
    # look one level down for the actual root
    candidates = [p for p in BASE_DIR.iterdir() if p.is_dir()]
    if candidates:
        BASE_DIR = candidates[0]
        print("Adjusted BASE_DIR to:", BASE_DIR)
 
BF_TRAIN = BASE_DIR / 'BF' / 'train'
BF_TEST  = BASE_DIR / 'BF' / 'test'
FL_TRAIN = BASE_DIR / 'FL' / 'train'
FL_TEST  = BASE_DIR / 'FL' / 'test'
 
TRAIN_CSV  = BASE_DIR / 'train.csv'
SAMPLE_SUB = BASE_DIR / 'sampleSubmission.csv'
OUT_DIR    = Path('/kaggle/working')
 
# Sanity check
for p, name in [(BF_TRAIN, 'BF/train'), (FL_TRAIN, 'FL/train'),
                (TRAIN_CSV, 'train.csv'), (SAMPLE_SUB, 'sampleSubmission.csv')]:
    status = "OK" if Path(p).exists() else "NOT FOUND"
    print(f"  {name}: {status}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
 
CFG = dict(
    img_size     = 224,
    batch_size   = 64,
    n_epochs     = 20,
    lr           = 3e-4,
    weight_decay = 1e-2,
    n_folds      = 5,
    label_smooth = 0.05,
    mixup_alpha  = 0.4,
    tta_n        = 5,
    amp          = True,
)

Competition files downloaded to: /kaggle/input/competitions/multimodal-cancer-classification-challenge-2026
  BF/train: OK
  FL/train: OK
  train.csv: OK
  sampleSubmission.csv: OK


In [5]:
train_df = pd.read_csv(TRAIN_CSV)
train_df['patient'] = train_df['Name'].str.extract(r'(pat_\d+)')
print(train_df.shape)
print(train_df['Diagnosis'].value_counts())
print(train_df['patient'].nunique(), 'unique patients')
 
pat_label = train_df.groupby('patient')['Diagnosis'].first().reset_index()
pat_label.columns = ['patient', 'label']
train_df = train_df.merge(pat_label, on='patient')
 
gkf = GroupKFold(n_splits=CFG['n_folds'])
train_df['fold'] = -1
for fold, (_, val_idx) in enumerate(
        gkf.split(train_df, train_df['label'], groups=train_df['patient'])):
    train_df.loc[val_idx, 'fold'] = fold
 
print(train_df['fold'].value_counts().sort_index())

(114302, 3)
Diagnosis
0    70000
1    44302
Name: count, dtype: int64
12 unique patients
fold
0    30000
1    24302
2    20000
3    20000
4    20000
Name: count, dtype: int64


In [6]:
def get_augmentation(mode='train', img_size=224):
    size = (img_size, img_size)   # newer albumentations requires a tuple
    if mode == 'train':
        return A.Compose([
            A.RandomResizedCrop(size=size, scale=(0.7, 1.0)),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2,
                          saturation=0.2, hue=0.05, p=0.5),
            A.GaussianBlur(blur_limit=(3, 7), p=0.2),
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    else:
        ops = [A.Resize(height=img_size, width=img_size)]
        if mode == 'tta':
            ops += [
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
            ]
        ops += [
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ]
        return A.Compose(ops)

In [7]:
class MultimodalCellDataset(Dataset):
    def __init__(self, df, bf_root, fl_root, transform=None, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.bf_root   = Path(bf_root)
        self.fl_root   = Path(fl_root)
        self.transform = transform
        self.is_test   = is_test
 
    def __len__(self):
        return len(self.df)
 
    def _load_bf(self, name):
        img = cv2.imread(str(self.bf_root / name))
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
 
    def _load_fl(self, name):
        stem = Path(name).stem
        p = None
        for ext in ['.jpg', '.png', '.tif', '.tiff']:
            candidate = self.fl_root / (stem + ext)
            if candidate.exists():
                p = candidate
                break
        if p is None:
            p = self.fl_root / name  # fallback
        img = np.array(Image.open(str(p)))
        if img.ndim == 2:
            img = np.stack([img] * 4, axis=-1)
        elif img.shape[2] == 3:
            extra = img.mean(axis=2, keepdims=True).astype(np.uint8)
            img = np.concatenate([img, extra], axis=-1)
        elif img.shape[2] > 4:
            img = img[:, :, :4]
        return img
 
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        name = row['Name']
        bf   = self._load_bf(name)
        fl   = self._load_fl(name)
 
        h, w = bf.shape[:2]
        fl   = cv2.resize(fl, (w, h))
 
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
 
        if self.transform:
            bf_t = self.transform(image=bf)['image']           # 3xHxW
 
            fl3   = fl[:, :, :3]
            fl4ch = fl[:, :, 3]
            fl3_t = self.transform(image=fl3)['image']         # 3xHxW (ImageNet norm)
            fl3_t = fl3_t * std + mean                         # back to 0-1
 
            sz    = CFG['img_size']
            fl4_r = cv2.resize(fl4ch, (sz, sz))
            fl4_t = torch.from_numpy(fl4_r.astype(np.float32) / 255.0).unsqueeze(0)
 
            fl_t  = torch.cat([fl3_t, fl4_t], dim=0)          # 4xHxW
        else:
            bf_t = torch.from_numpy(bf.astype(np.float32) / 255.).permute(2, 0, 1)
            fl_t = torch.from_numpy(fl.astype(np.float32) / 255.).permute(2, 0, 1)
 
        if self.is_test:
            return bf_t, fl_t, name
 
        label   = torch.tensor(row['label'], dtype=torch.float32)
        patient = row['patient']
        return bf_t, fl_t, label, patient
 

In [8]:
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, downsample=None):
        super().__init__()
        self.conv1      = nn.Conv2d(in_ch, out_ch, 3, stride=stride,
                                    padding=1, bias=False)
        self.bn1        = nn.BatchNorm2d(out_ch)
        self.conv2      = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2        = nn.BatchNorm2d(out_ch)
        self.downsample = downsample
        self.relu       = nn.ReLU(inplace=True)
 
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample:
            identity = self.downsample(x)
        return self.relu(out + identity)
 
 
class ResNet14Stream(nn.Module):
    def __init__(self, in_channels=3, base_width=64):
        super().__init__()
        w = base_width
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, w, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(w), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1  = self._make_layer(w,     w,   2, stride=1)
        self.layer2  = self._make_layer(w,     w*2, 2, stride=2)
        self.layer3  = self._make_layer(w*2,   w*4, 2, stride=2)
        self.out_dim = w * 4
 
    def _make_layer(self, in_ch, out_ch, blocks, stride):
        ds = None
        if stride != 1 or in_ch != out_ch:
            ds = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        layers = [BasicBlock(in_ch, out_ch, stride=stride, downsample=ds)]
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_ch, out_ch))
        return nn.Sequential(*layers)
 
    def forward(self, x):
        return self.layer3(self.layer2(self.layer1(self.stem(x))))
 
 
class CrossModalAttentionGate(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Conv2d(channels, channels // 4, 1, bias=False),
            nn.BatchNorm2d(channels // 4),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 4, channels, 1, bias=False),
            nn.Sigmoid(),
        )
        self.bn = nn.BatchNorm2d(channels)
 
    def forward(self, bf_feat, fl_feat):
        g = self.gate(fl_feat)
        return self.bn(bf_feat * g + bf_feat)
 
 
class AttentionMILHead(nn.Module):
    def __init__(self, in_dim, hidden=128):
        super().__init__()
        self.V   = nn.Linear(in_dim, hidden)
        self.U   = nn.Linear(in_dim, hidden)
        self.w   = nn.Linear(hidden, 1)
        self.cls = nn.Linear(in_dim, 1)
 
    def forward(self, h):
        a   = self.w(torch.tanh(self.V(h)) * torch.sigmoid(self.U(h)))
        a   = torch.softmax(a, dim=0)
        bag = (a * h).sum(dim=0)
        return self.cls(bag).squeeze(-1)
 
 
class DualStreamCancerModel(nn.Module):
    def __init__(self, base_width=64):
        super().__init__()
        self.bf_enc    = ResNet14Stream(in_channels=3, base_width=base_width)
        self.fl_enc    = ResNet14Stream(in_channels=4, base_width=base_width)
        C              = self.bf_enc.out_dim
        self.cross_attn = CrossModalAttentionGate(C)
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.cell_proj  = nn.Sequential(
            nn.Linear(C * 2, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
        )
        self.cell_head = nn.Linear(256, 1)
        self.mil_head  = AttentionMILHead(256, hidden=128)
 
    def encode(self, bf, fl):
        bf_f = self.bf_enc(bf)
        fl_f = self.fl_enc(fl)
        bf_g = self.cross_attn(bf_f, fl_f)
        bf_v = self.pool(bf_g).flatten(1)
        fl_v = self.pool(fl_f).flatten(1)
        return self.cell_proj(torch.cat([bf_v, fl_v], dim=1))
 
    def forward(self, bf, fl):
        return self.cell_head(self.encode(bf, fl)).squeeze(-1)

In [9]:
class LabelSmoothBCE(nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.s = smoothing
 
    def forward(self, logits, targets):
        targets = targets * (1 - self.s) + 0.5 * self.s
        return F.binary_cross_entropy_with_logits(logits, targets)
 
 
def mixup_data(bf, fl, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(bf.size(0), device=bf.device)
    return (lam * bf + (1 - lam) * bf[idx],
            lam * fl + (1 - lam) * fl[idx],
            lam * y  + (1 - lam) * y[idx])

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scaler):
  
    model.train()
    total_loss = 0.0

    for bf, fl, label, patients in loader:
        bf    = bf.to(DEVICE)
        fl    = fl.to(DEVICE)
        label = label.to(DEVICE)

        # per-cell loss with mixup
        bf_m, fl_m, label_m = mixup_data(bf, fl, label, alpha=CFG['mixup_alpha'])
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=CFG['amp']):
            cell_logits = model(bf_m, fl_m)
            cell_loss   = criterion(cell_logits, label_m)

            # MIL loss — group cells in this batch by patient
            embeddings = model.encode(bf, fl)   # (B, 256)
            mil_logits = []
            mil_labels = []
            unique_pats = list(dict.fromkeys(patients))  # preserve order
            for pat in unique_pats:
                mask = [i for i, p in enumerate(patients) if p == pat]
                if len(mask) < 2:     # skip lone cells
                    continue
                h     = embeddings[mask]   # (n, 256)
                logit = model.mil_head(h)  # scalar
                mil_logits.append(logit)
                mil_labels.append(label[mask[0]])

            if mil_logits:
                mil_logits = torch.stack(mil_logits)
                mil_labels = torch.stack(mil_labels)
                mil_loss   = F.binary_cross_entropy_with_logits(
                                 mil_logits, mil_labels)
                loss = cell_loss + 0.5 * mil_loss
            else:
                loss = cell_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    return total_loss / len(loader)
 
 
@torch.no_grad()
def validate_patient_level(model, val_df, bf_root, fl_root):
    model.eval()
    transform = get_augmentation('val', CFG['img_size'])

    patient  = val_df['patient'].iloc[0]
    true_lbl = val_df['label'].iloc[0]

    ds  = MultimodalCellDataset(val_df, bf_root, fl_root, transform=transform)
    dl  = DataLoader(ds, batch_size=CFG['batch_size'], shuffle=False,
                     num_workers=2, pin_memory=True)
    embs = []
    for bf, fl, _, _ in dl:
        bf, fl = bf.to(DEVICE), fl.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CFG['amp']):
            embs.append(model.encode(bf, fl).cpu())

    H     = torch.cat(embs, dim=0).to(DEVICE)
    score = torch.sigmoid(model.mil_head(H)).cpu().item()

    # Can't compute AUC with 1 patient — return score + label for later aggregation
    print(f"    val patient={patient} true={true_lbl} pred={score:.4f}")
    return score, true_lbl


# New run_fold that collects score+label instead of per-fold AUC
def run_fold(fold, train_df, bf_train, fl_train):
    sep = "-" * 55
    print(f"\n{sep}\n  FOLD {fold} — val patient: "
          f"{train_df[train_df['fold']==fold]['patient'].iloc[0]}\n{sep}")

    tr_df  = train_df[train_df['fold'] != fold]
    val_df = train_df[train_df['fold'] == fold]

    tr_ds = MultimodalCellDataset(
        tr_df, bf_train, fl_train,
        transform=get_augmentation('train', CFG['img_size']))
    tr_dl = DataLoader(tr_ds, batch_size=CFG['batch_size'], shuffle=True,
                       num_workers=4, pin_memory=True, drop_last=True)

    model     = DualStreamCancerModel(base_width=64).to(DEVICE)
    criterion = LabelSmoothBCE(CFG['label_smooth'])
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG['n_epochs'], eta_min=1e-6)
    scaler    = torch.cuda.amp.GradScaler(enabled=CFG['amp'])

    best_score = -1
    best_label = -1
    best_ckpt  = OUT_DIR / f'model_fold{fold}.pth'

    for epoch in range(1, CFG['n_epochs'] + 1):
        tr_loss = train_epoch(model, tr_dl, optimizer, criterion, scaler)
        scheduler.step()

        if epoch % 2 == 0 or epoch == CFG['n_epochs']:
            score, true_lbl = validate_patient_level(
                model, val_df, bf_train, fl_train)
            print(f"  Epoch {epoch:02d} | loss {tr_loss:.4f} | "
                  f"pred={score:.4f} true={true_lbl}")
            if best_score < 0 or epoch == CFG['n_epochs']:
                best_score = score
                best_label = true_lbl
                torch.save(model.state_dict(), best_ckpt)
        else:
            print(f"  Epoch {epoch:02d} | loss {tr_loss:.4f}")

    del model, tr_dl, tr_ds
    gc.collect()
    torch.cuda.empty_cache()
    return best_score, best_label, str(best_ckpt)


# Run all folds and compute LOPO-AUC at the end
fold_results = []
for fold in range(CFG['n_folds']):
    score, label, ckpt = run_fold(fold, train_df, BF_TRAIN, FL_TRAIN)
    fold_results.append({'fold': fold, 'score': score,
                         'label': label, 'ckpt': ckpt})

# NOW compute AUC across all 12 patients
all_preds  = [r['score'] for r in fold_results]
all_labels = [r['label'] for r in fold_results]
lopo_auc   = roc_auc_score(all_labels, all_preds)

print("\nLOPO-CV Results (per patient):")
for r in fold_results:
    pat = train_df[train_df['fold']==r['fold']]['patient'].iloc[0]
    print(f"  {pat}  true={r['label']}  pred={r['score']:.4f}")
print(f"\nFinal LOPO-AUC: {lopo_auc:.4f}")


-------------------------------------------------------
  FOLD 0 — val patient: pat_03
-------------------------------------------------------


In [12]:
print("Total rows:", len(train_df))
print("Unique patients:", train_df['patient'].nunique())
print("\nLabel distribution:")
print(train_df.groupby('patient')['label'].first().value_counts())
print("\nFold distribution:")
print(train_df.groupby('fold')['patient'].nunique())
print("\nAny patient in multiple folds?")
pat_fold = train_df.groupby('patient')['fold'].nunique()
print("Patients spanning >1 fold:", (pat_fold > 1).sum())  # must be 0
print("\nSample rows:")
print(train_df[['Name','patient','label','fold']].head(10))

Total rows: 114302
Unique patients: 12

Label distribution:
label
0    7
1    5
Name: count, dtype: int64

Fold distribution:
fold
0    3
1    3
2    2
3    2
4    2
Name: patient, dtype: int64

Any patient in multiple folds?
Patients spanning >1 fold: 0

Sample rows:
                  Name patient  label  fold
0   pat_03_image_1.jpg  pat_03      1     0
1   pat_03_image_2.jpg  pat_03      1     0
2   pat_03_image_3.jpg  pat_03      1     0
3   pat_03_image_4.jpg  pat_03      1     0
4   pat_03_image_5.jpg  pat_03      1     0
5   pat_03_image_6.jpg  pat_03      1     0
6   pat_03_image_7.jpg  pat_03      1     0
7   pat_03_image_8.jpg  pat_03      1     0
8   pat_03_image_9.jpg  pat_03      1     0
9  pat_03_image_10.jpg  pat_03      1     0


In [11]:
def run_fold(fold, train_df, bf_train, fl_train):
    sep = "-" * 55
    print(f"\n{sep}\n  FOLD {fold}\n{sep}")
 
    tr_df  = train_df[train_df['fold'] != fold]
    val_df = train_df[train_df['fold'] == fold]
 
    tr_ds = MultimodalCellDataset(
        tr_df, bf_train, fl_train,
        transform=get_augmentation('train', CFG['img_size']))
    tr_dl = DataLoader(tr_ds, batch_size=CFG['batch_size'], shuffle=True,
                       num_workers=4, pin_memory=True, drop_last=True)
 
    model     = DualStreamCancerModel(base_width=64).to(DEVICE)
    criterion = LabelSmoothBCE(CFG['label_smooth'])
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=CFG['lr'],
                                  weight_decay=CFG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG['n_epochs'], eta_min=1e-6)
    scaler    = torch.cuda.amp.GradScaler(enabled=CFG['amp'])
 
    best_auc  = 0.0
    best_ckpt = OUT_DIR / f'model_fold{fold}.pth'
 
    for epoch in range(1, CFG['n_epochs'] + 1):
        tr_loss = train_epoch(model, tr_dl, optimizer, criterion, scaler)
        scheduler.step()
 
        if epoch % 2 == 0 or epoch == CFG['n_epochs']:
            auc, _, _ = validate_patient_level(model, val_df, bf_train, fl_train)
            print(f"  Epoch {epoch:02d} | loss {tr_loss:.4f} | patient-AUC {auc:.4f}")
            if auc > best_auc:
                best_auc = auc
                torch.save(model.state_dict(), best_ckpt)
                print(f"  -> Saved best (AUC={best_auc:.4f})")
        else:
            print(f"  Epoch {epoch:02d} | loss {tr_loss:.4f}")
 
    del model, tr_dl, tr_ds
    gc.collect()
    torch.cuda.empty_cache()
    return best_auc, str(best_ckpt)
 
 
fold_results = []
for fold in range(CFG['n_folds']):
    auc, ckpt = run_fold(fold, train_df, BF_TRAIN, FL_TRAIN)
    fold_results.append({'fold': fold, 'auc': auc, 'ckpt': ckpt})
 
print("\nCV Summary")
aucs = [r['auc'] for r in fold_results]
for r in fold_results:
    print(f"  Fold {r['fold']}: AUC = {r['auc']:.4f}")
print(f"  Mean {np.mean(aucs):.4f}  Std {np.std(aucs):.4f}")


-------------------------------------------------------
  FOLD 0
-------------------------------------------------------


Corrupt JPEG data: 29 extraneous bytes before marker 0xd9


  Epoch 01 | loss 0.5499


Corrupt JPEG data: 29 extraneous bytes before marker 0xd9


  Epoch 02 | loss 0.4863 | patient-AUC 1.0000
  -> Saved best (AUC=1.0000)


Corrupt JPEG data: 29 extraneous bytes before marker 0xd9


  Epoch 03 | loss 0.4620


Corrupt JPEG data: 29 extraneous bytes before marker 0xd9


KeyboardInterrupt: 

In [ ]:
sample_sub = pd.read_csv(SAMPLE_SUB)
if 'patient' not in sample_sub.columns:
    sample_sub['patient'] = sample_sub['Name'].str.extract(r'(pat_\d+)')
test_patients = sample_sub['patient'].unique()
print(f'{len(test_patients)} test patients')
 
 
@torch.no_grad()
def predict_patients(model, test_patients, bf_test, fl_test, tta_n=5):
    model.eval()
    preds = {}
    for patient in test_patients:
        pat_files = sorted(Path(bf_test).glob(f'{patient}_image_*.jpg'))
        if not pat_files:
            pat_files = sorted(Path(bf_test).glob(f'{patient}*.jpg'))
        df_pat = pd.DataFrame({
            'Name': [p.name for p in pat_files],
            'label': 0,
            'patient': patient,
        })
 
        tta_embs = []
        for _ in range(tta_n):
            ds  = MultimodalCellDataset(
                df_pat, bf_test, fl_test,
                transform=get_augmentation('tta', CFG['img_size']),
                is_test=True)
            dl  = DataLoader(ds, batch_size=CFG['batch_size'], shuffle=False,
                             num_workers=2, pin_memory=True)
            embs = []
            for bf, fl, _ in dl:
                bf, fl = bf.to(DEVICE), fl.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=CFG['amp']):
                    embs.append(model.encode(bf, fl).cpu())
            tta_embs.append(torch.cat(embs, dim=0))
 
        H     = torch.stack(tta_embs).mean(dim=0).to(DEVICE)
        logit = model.mil_head(H).cpu().item()
        preds[patient] = torch.sigmoid(torch.tensor(logit)).item()
    return preds
 
 
all_fold_preds = defaultdict(list)
for r in fold_results:
    model = DualStreamCancerModel(base_width=64).to(DEVICE)
    model.load_state_dict(torch.load(r['ckpt'], map_location=DEVICE))
    fp = predict_patients(model, test_patients, BF_TEST, FL_TEST,
                          tta_n=CFG['tta_n'])
    for pat, prob in fp.items():
        all_fold_preds[pat].append(prob)
    del model
    gc.collect()
    torch.cuda.empty_cache()
 
final_preds = {pat: float(np.mean(probs))
               for pat, probs in all_fold_preds.items()}